# ECB Supervisory Guide → JSON (Google Colab)

This notebook turns every paragraph of the [ECB guide to internal models / supervisory guide (July 2025)](https://www.bankingsupervision.europa.eu/ecb/pub/pdf/ssm.supervisory_guide202507.en.pdf) into one JSON record using a **pure standard-library** PDF parser — no `pip install` required.

Each record looks like:
```json
{
  "id": 1,
  "paragraph_number": "1",
  "chapter": "Foreword",
  "section": "5.4 Data quality",
  "page_start": 6,
  "page_end": 6,
  "text": "1. Articles 143, 283 and 325 of Regulation (EU) No 575/2013 (CRR)…"
}
```

**How to run:** `Runtime → Run all`. The steps are:
1. Clone the repo (it already bundles the parser **and** the PDF).
2. Run the parser.
3. Preview the result.
4. Download the JSON.

If your repo is private, see the *"Upload files manually"* fallback at the bottom.

## 1. Get the code and the PDF

Clones the feature branch, which contains `pdf_extract.py`, `ecb_pdf_to_json.py` and the source PDF.

In [ ]:
import os

REPO   = "https://github.com/bryanlumadjeng/ecb_json_parser.git"
BRANCH = "claude/paragraph-json-transpose-f8lsqh"
WORKDIR = "/content/ecb_json_parser"

if not os.path.isdir(WORKDIR):
    !git clone --depth 1 --branch "$BRANCH" "$REPO" "$WORKDIR"

os.chdir(WORKDIR)
print("Working directory:", os.getcwd())
!ls -la

## 2. (Optional) Higher-fidelity layout with PyMuPDF

The parser runs fine on the standard library alone. Installing PyMuPDF is **optional** — if present it is used automatically for slightly better layout fidelity. Uncomment to enable.

In [ ]:
# !pip install -q pymupdf

## 3. Run the parser

Reads the bundled PDF and writes `supervisory_guide_paragraphs.json`.

In [ ]:
PDF    = "ssm.supervisory_guide202507.en.pdf"
OUTPUT = "supervisory_guide_paragraphs.json"

!python ecb_pdf_to_json.py --pdf "$PDF" -o "$OUTPUT"

## 4. Preview the result

In [ ]:
import json

with open(OUTPUT, encoding="utf-8") as fh:
    data = json.load(fh)

paragraphs = data["paragraphs"]
print(f"source        : {data['source']}")
print(f"extracted_at  : {data['extracted_at']}")
print(f"paragraphs    : {data['paragraph_count']}")
print(f"numbered ones : {sum(1 for p in paragraphs if p['paragraph_number'])}")

print("\n--- first few numbered paragraphs ---")
shown = 0
for p in paragraphs:
    if p["paragraph_number"]:
        print(f"\n[{p['paragraph_number']}] {p['chapter']} — {p['section']}  (p.{p['page_start']})")
        print(p["text"][:300])
        shown += 1
        if shown >= 3:
            break

### Browse as a table (pandas)

In [ ]:
import pandas as pd

df = pd.DataFrame(paragraphs)
df.head(20)

## 5. Download the JSON

In [ ]:
from google.colab import files

files.download(OUTPUT)

---
## Fallback: upload files manually

Use this only if the `git clone` step failed (e.g. the repo is private and not authenticated in Colab). Upload `pdf_extract.py`, `ecb_pdf_to_json.py` and your PDF, then re-run **section 3** onward.

You can also use this to parse a **different** PDF: upload it, then set `PDF = "your_file.pdf"` in section 3.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick pdf_extract.py, ecb_pdf_to_json.py and/or a PDF
print("Uploaded:", list(uploaded))